### Setup

In [1]:
import numpy as np
import choix
from scipy.optimize import minimize
import scipy.stats as stats
import matplotlib.pyplot as plt
import random
from matplotlib import colors
import pandas as pd
import seaborn as sns
import pickle
import os
import sys

sys.path.append(os.path.abspath('../../'))
from metrics import compute_acc, compute_weighted_acc
from opt_fair import *
from distribution_utils import safe_kendalltau, to_numpy

In [2]:
!nvidia-smi

Sat Jun 27 05:29:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:17:00.0 Off |                    0 |
| N/A   73C    P0            269W /  300W |   29873MiB /  81920MiB |    100%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print(device)

cuda


In [4]:
print(f"Current PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

Current PyTorch version: 2.4.0a0+f70bd71a48.nv24.06
CUDA available: True
CUDA version: 12.5


In [5]:
with open("data/PassagePC.pickle", 'rb') as handle:
    PC_faceage = pickle.load(handle)    
with open("data/PassageDF.pickle", 'rb') as handle:
    df_faceage = pickle.load(handle)

In [6]:
df_faceage

,label,score
0,"a star. Our planet, Earth, orbits, or circles,...",1
1,"Adam, We did not have plastic toys. I played w...",1
2,Who said the little owl. Who wants to hunt wit...,1
3,dead leaf. This is a mole. Moles burrow underg...,1
4,ereaddatagradepsenvironcomp.html Environment r...,1
...,...,...
467,work over the summer on any changes they wish ...,12
468,between January and December plunged the Unite...,12
469,into a newly opened bank account. I was amazed...,12
470,"occurring phenomenon, manmade by products are ...",12


In [7]:
import opt_fair
all_pc_faceage  = opt_fair._pc_without_reviewers(PC_faceage)

size = len(df_faceage)
print(size)
classes = [0]*size

472


In [8]:
print(len(all_pc_faceage))

11763


In [9]:
gt_scores = to_numpy(df_faceage['score'].tolist())

In [10]:
max_iter=30000

In [11]:
lr = 0.001

In [12]:
tol = 1e-6

### HTCV

In [13]:
import random
from htcv_m import *

In [14]:
import time
Grad_accs, Grad_waccs, Grad_taus = [], [], []
gradem_time = []
for sd in range(1):
    start = time.time()
    grad_em = HTCVWrapper(PC_faceage, lr, sd, device, max_iter=max_iter)
    r_est, beta_est = grad_em.run_algorithm()
    end = time.time()
    gradem_time.append(end-start)

    r_est_np = to_numpy(r_est)
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    grad_acc = compute_acc(df_faceage, 1*r_est_np, device)
    grad_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    grad_tau = safe_kendalltau(r_est_np, gt_scores)
    
    Grad_accs.append(grad_acc)    
    Grad_waccs.append(grad_wacc)    
    Grad_taus.append(grad_tau)

 20%|███████████████▍                                                             | 6014/30000 [00:33<02:12, 181.51it/s]



Converged at epoch 6014


In [15]:
print(f"{np.mean(gradem_time)} +- {np.std(gradem_time)}")

34.766416788101196 +- 0.0


In [16]:
print(f"HTCV -- Accuracy: {np.mean(Grad_accs)} ± {np.std(Grad_accs)}, Weighted Accuracy: {np.mean(Grad_waccs)} ± {np.std(Grad_waccs)}, Kendall's Tau: {np.mean(Grad_taus)} ± {np.std(Grad_taus)}")

HTCV -- Accuracy: 0.6929646730422974 ± 0.0, Weighted Accuracy: 0.7530044913291931 ± 0.0, Kendall's Tau: 0.3643207415822173 ± 0.0


### HBTL

In [17]:
import random
from grad_em import *

In [18]:
import time
Grad_accs, Grad_waccs, Grad_taus = [], [], []
gradem_time = []
for sd in range(1):
    start = time.time()
    grad_em = GradientEMWrapper(PC_faceage, lr, sd, device, max_iter=max_iter)
    r_est, beta_est = grad_em.run_algorithm()
    end = time.time()
    gradem_time.append(end-start)

    r_est_np = to_numpy(r_est)
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    grad_acc = compute_acc(df_faceage, 1*r_est_np, device)
    grad_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    grad_tau = safe_kendalltau(r_est_np, gt_scores)
    
    Grad_accs.append(grad_acc)    
    Grad_waccs.append(grad_wacc)    
    Grad_taus.append(grad_tau)

 40%|██████████████████████████████                                              | 11856/30000 [00:46<01:11, 253.14it/s]


In [19]:
print(f"{np.mean(gradem_time)} +- {np.std(gradem_time)}")

46.88823914527893 +- 0.0


In [20]:
print(f"HBTL -- Accuracy: {np.mean(Grad_accs)} ± {np.std(Grad_accs)}, Weighted Accuracy: {np.mean(Grad_waccs)} ± {np.std(Grad_waccs)}, Kendall's Tau: {np.mean(Grad_taus)} ± {np.std(Grad_taus)}")

HBTL -- Accuracy: 0.6953168511390686 ± 0.0, Weighted Accuracy: 0.7545710802078247 ± 0.0, Kendall's Tau: 0.3687617017322948 ± 0.0


### PG EM

In [21]:
import random
from pgem_initialize_beta_1 import EMWrapper, EMWrapper_3_0

In [22]:
PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []
pgem_time = []
for sd in range(1):
    start = time.time()
    pg = EMWrapper(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    pgem_time.append(end-start)

    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda
cuda


  0%|                                                                               | 1/30000 [00:00<4:39:01,  1.79it/s]

Iter 000: Log-likelihood = -0.461260


  0%|▏                                                                             | 59/30000 [00:24<3:31:26,  2.36it/s]

Converged at iter 59, Log-likelihood change = 8.940697e-07


In [23]:
print(f"{np.mean(pgem_time)} +- {np.std(pgem_time)}")

25.034984350204468 +- 0.0


In [24]:
PGEM_accs

[0.6958821415901184]

In [25]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.6958821415901184 ± 0.0, Weighted Accuracy: 0.7555873990058899 ± 0.0, Kendall's Tau: 0.3698290569614979 ± 0.0


### Faster PGEM

In [26]:
import time

PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []

for sd in range(1):
    start = time.time()
    pg = EMWrapper_3_0(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    print("Elapsed:", end - start, "seconds")
    
    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda


  0%|                                                                                 | 2/30000 [00:00<27:47, 17.99it/s]

Iter 000: Log-likelihood = -0.470411


  0%|                                                                                | 23/30000 [00:01<30:55, 16.15it/s]

Converged at iter 23, ΔLL = 8.940697e-07
Elapsed: 1.4688293933868408 seconds


In [27]:
PGEM_accs

[0.696699857711792]

In [28]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.696699857711792 ± 0.0, Weighted Accuracy: 0.7567439079284668 ± 0.0, Kendall's Tau: 0.3713729100608811 ± 0.0


### BT

In [13]:
%%time
bt_scores = opt_fair.opt_pairwise_gpu(size, all_pc_faceage, alpha=0, initial_params=None, max_iter=max_iter, tol=tol)

BT finished
CPU times: user 2.55 s, sys: 559 ms, total: 3.11 s
Wall time: 3.21 s


In [14]:
r_est_np = to_numpy(bt_scores)
bt_acc = compute_acc(df_faceage, 1*r_est_np, device)
bt_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
bt_tau = safe_kendalltau(r_est_np, gt_scores)

In [15]:
print(f"Simple BT -- Accuracy: {bt_acc}, Weighted Accuracy: {bt_wacc}, Kendall's Tau: {bt_tau} ")

Simple BT -- Accuracy: 0.6803860664367676, Weighted Accuracy: 0.7432994246482849, Kendall's Tau: 0.3407005748100444 


### BARP

In [16]:
FaceAge = opt_fair.BARP(data = PC_faceage, penalty = 0, classes = classes, device=device)

In [17]:
crowd_labels = pd.read_csv('data/passage_cleaned.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [18]:
%%time
annot_bt_temp, annot_bias =  opt_fair._alternate_optim_torch(size, num_reviewers, FaceAge, lr_x=lr, lr_y=lr, iters = max_iter)

 21%|████████████████                                                             | 6280/30000 [00:30<01:55, 205.76it/s]

CPU times: user 29.8 s, sys: 1.05 s, total: 30.9 s
Wall time: 30.5 s


In [19]:
r_est_np = to_numpy(annot_bt_temp)
gt_scores = to_numpy(df_faceage['score'].tolist())
barp_acc = compute_acc(df_faceage, 1*r_est_np, device)
barp_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
barp_tau = safe_kendalltau(r_est_np, gt_scores)

In [20]:
print(f"BARP -- Accuracy: {barp_acc}, Weighted Accuracy: {barp_wacc}, Kendall's Tau: {barp_tau} ")

BARP -- Accuracy: 0.6819003224372864, Weighted Accuracy: 0.7450876235961914, Kendall's Tau: 0.3434864501515187 


### RC

In [21]:
%%time

rc_obj = RankCentrality(device)
A = rc_obj.matrix_of_comparisons(size, all_pc_faceage)
# print("A matrix done")
P = rc_obj.trans_prob(A)
# print("P matrix done")
pi = rc_obj.stationary_dist(P, max_iter=max_iter, tol=tol)
rank_centrality_scores = torch.log(pi).cpu().numpy()

RC converged at iteration 3
RC finished
CPU times: user 87.3 ms, sys: 73.8 ms, total: 161 ms
Wall time: 157 ms


In [22]:
r_est_np = to_numpy(rank_centrality_scores)
rc_acc = compute_acc(df_faceage, 1*r_est_np, device)
rc_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
rc_tau = safe_kendalltau(r_est_np, gt_scores)

In [23]:
print(f"RC -- Accuracy: {rc_acc}, Weighted Accuracy: {rc_wacc}, Kendall's Tau: {rc_tau} ")

RC -- Accuracy: 0.6521800756454468, Weighted Accuracy: 0.71303790807724, Kendall's Tau: 0.289180935098035 


### CrowdBT

In [13]:
import time
crowd_labels = pd.read_csv('data/passage_cleaned.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [14]:
print(device)
gt_scores = to_numpy(df_faceage['score'].tolist())

cuda


In [15]:
CrowdBT_accs, CrowdBT_waccs, CrowdBT_taus = [], [], []
K = num_reviewers
gt_df = df_faceage
crowdbt_time = []
for seed in range(1):
    try:
        start = time.time()
        crowdbt_test = opt_fair.CrowdBT_3_0(data=PC_faceage, penalty=0, device=device, random_seed=seed)
        crowdbt_scores, _ = crowdbt_test.alternate_optim(size, K, lr_x=lr, lr_y=lr, iters=max_iter)
        end = time.time()
        crowdbt_time.append(end-start)
        r_est_np = to_numpy(crowdbt_scores)
        gt_scores = to_numpy(df_faceage['score'].tolist())
        current_tau = safe_kendalltau(r_est_np, gt_scores)
        if current_tau < 0:
            r_est_np = -r_est_np
        crowdbt_acc = compute_acc(df_faceage, 1*r_est_np, device)
        crowdbt_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
        crowdbt_tau = safe_kendalltau(r_est_np, gt_scores)
        CrowdBT_accs.append(crowdbt_acc)
        CrowdBT_waccs.append(crowdbt_wacc)
        CrowdBT_taus.append(crowdbt_tau)
    except Exception as e:
        print(e)
        CrowdBT_accs.append(0.0)
        CrowdBT_waccs.append(0.0)
        CrowdBT_taus.append(0.0)

 46%|████████████████████████████▊                                 | 13933/30000 [01:09<01:20, 199.44it/s, loss=4.97e+3]


In [16]:
print(f"{np.mean(crowdbt_time)} +- {np.std(crowdbt_time)}")

71.41738057136536 +- 0.0


In [17]:
print(f"CrowdBT -- Accuracy: {np.mean(CrowdBT_accs)} ± {np.std(CrowdBT_accs)}, Weighted Accuracy: {np.mean(CrowdBT_waccs)} ± {np.std(CrowdBT_waccs)}, Kendall's Tau: {np.mean(CrowdBT_taus)} ± {np.std(CrowdBT_taus)}")

CrowdBT -- Accuracy: 0.7001726031303406 ± 0.0, Weighted Accuracy: 0.7595662474632263 ± 0.0, Kendall's Tau: 0.37792952075455777 ± 0.0


### FactorBT

In [18]:
crowd_labels = pd.read_csv('data/passage_cleaned.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [19]:
print(device)
gt_scores = to_numpy(df_faceage['score'].tolist())

cuda


In [20]:
from collections import defaultdict
from pathlib import Path

def build_pc_dict_for_noisybt(df, df_faceage, worker_col="worker", left_col="left", right_col="right", label_col="label"):
    """
    Builds the dict NoisyBT_3_0 expects: {worker_id: [(left, right, winner), ...]}

    Mirrors the exact item/worker id mapping used to build PC_faceage_spm:
      - worker ids: position of first appearance in df[worker_col].unique()
      - item ids: position of df_faceage['full_path'] (matched via Path(...).name,
        since df['left']/df['right']/df['label'] are full URLs but full_path
        entries are basenames)

    Unlike PC_faceage_spm (which collapses each row to (winner, loser)), this
    keeps the left/right slot assignment, since NoisyBT's bias parameter needs
    to know which item was shown in the "left" position.

    Returns the dict AND the two id mappings, so you can invert them later
    if needed (e.g. to map scores back to filenames).
    """
    unique_performers = list(df[worker_col].unique())
    performer_label_dict = {performer: i for i, performer in enumerate(unique_performers)}

    item_labels = list(df_faceage["label"])
    item_label_dict = {item: i for i, item in enumerate(item_labels)}

    pc_dict = defaultdict(list)
    for performer, group in df.groupby(worker_col):
        key = performer_label_dict[performer]
        for _, row in group.iterrows():
            left = row[left_col]
            right = row[right_col]
            winner = row[label_col]
            
            left_id = item_label_dict[left]
            right_id = item_label_dict[right]
            winner_id = item_label_dict[winner]

            pc_dict[key].append((left_id, right_id, winner_id))

    return dict(pc_dict), performer_label_dict, item_label_dict

In [21]:
def sort_df(df, column_name):
    # Sort by a specific column (replace 'column_name' with your column)
    df_sorted = df.sort_values(by=column_name, ascending=True)  # or ascending=False

    return df_sorted

df = pd.read_csv("data/passage_cleaned.csv")
df = df.rename(columns={'performer': 'worker'})

In [22]:
df.head()

,Unnamed: 0,_unit_id,_created_at,_golden,_id,_missed,_started_at,_tainted,_trust,worker,please_decide_which_passage_is_more_difficult,label_level_a,label_level_b,left,right,please_decide_which_passage_is_more_difficult_gold,label
0,8371,101707741,9/22/2011 11:40,False,213246266,NaN,9/22/2011 11:36,True,0.5,5,Passage B is more difficult.,5,7,been wicked. They believed that the end of the...,lichen Sect Content Linking Artid A snake coil...,NaN,lichen Sect Content Linking Artid A snake coil...
1,10346,101707977,9/22/2011 11:40,False,213246261,NaN,9/22/2011 11:36,True,0.5,5,Passage A is more difficult.,9,8,primitive hunters surrounded the herds with fi...,"For one thing, in the second half of the thirt...",NaN,primitive hunters surrounded the herds with fi...
2,13442,101708316,9/22/2011 11:40,False,213246262,NaN,9/22/2011 11:36,True,0.5,5,Passage A is more difficult.,7,12,possible error HhexagonhistogramhypotenuseIIde...,while a simpler solution would be to use fuels...,NaN,possible error HhexagonhistogramhypotenuseIIde...
3,3673,101707232,9/22/2011 11:35,False,213244638,NaN,9/22/2011 11:32,True,0.5,5,Passage B is more difficult.,3,7,"to tell him how to go about catching fish. Oh,...",and homespun remedies to keep deer out of my g...,NaN,and homespun remedies to keep deer out of my g...
4,2004,101707077,9/22/2011 11:35,False,213244637,NaN,9/22/2011 11:32,True,0.5,5,Passage A is more difficult.,10,2,edict which prevented chaos was the following ...,cold it freezes. To freeze is to become very c...,NaN,edict which prevented chaos was the following ...


In [23]:
gt_df = df_faceage
gt_df.head()

,label,score
0,"a star. Our planet, Earth, orbits, or circles,...",1
1,"Adam, We did not have plastic toys. I played w...",1
2,Who said the little owl. Who wants to hunt wit...,1
3,dead leaf. This is a mole. Moles burrow underg...,1
4,ereaddatagradepsenvironcomp.html Environment r...,1


In [24]:
import time
NoisyBT_accs, NoisyBT_waccs, NoisyBT_taus = [], [], []
K = num_reviewers
gt_df = df_faceage
noisybt_time = []

# Build the (left, right, winner) dict NoisyBT_3_0 needs, from the same
# crowd_labels.csv-derived df used by the original NoisyBradleyTerry.
# `df` must have integer-coded 'worker', 'left', 'right', 'label' columns
# aligned with `size`/num_items and gt_df['score'] (see build_pc_dict.py).
PC_faceage_noisybt, _performer_label_dict, _item_label_dict = build_pc_dict_for_noisybt(df, df_faceage)

for seed in range(1):
    try:
        start = time.time()
        noisybt_test = opt_fair.NoisyBT_3_0(data=PC_faceage_noisybt, penalty=0, device=device, random_seed=seed)
        noisybt_scores, noisybt_skills, noisybt_biases = noisybt_test.alternate_optim(size, K, lr_x=lr, lr_y=lr, iters=max_iter)
        end = time.time()
        noisybt_time.append(end-start)
        r_est_np = to_numpy(noisybt_scores)
        gt_scores = to_numpy(gt_df['score'].tolist())
        current_tau = safe_kendalltau(r_est_np, gt_scores)
        if current_tau < 0:
            r_est_np = -r_est_np
        noisybt_acc = compute_acc(gt_df, 1*r_est_np, device)
        noisybt_wacc = compute_weighted_acc(gt_df, 1*r_est_np, device)
        noisybt_tau = safe_kendalltau(r_est_np, gt_scores)
        NoisyBT_accs.append(noisybt_acc)
        NoisyBT_waccs.append(noisybt_wacc)
        NoisyBT_taus.append(noisybt_tau)
    except Exception as e:
        print(e)
        NoisyBT_accs.append(0.0)
        NoisyBT_waccs.append(0.0)
        NoisyBT_taus.append(0.0)

 52%|████████████████████████████████▏                             | 15604/30000 [01:32<01:25, 168.60it/s, loss=4.71e+3]


In [25]:
print(f"FactorBT -- Accuracy: {np.mean(NoisyBT_accs)} ± {np.std(NoisyBT_accs)}, Weighted Accuracy: {np.mean(NoisyBT_waccs)} ± {np.std(NoisyBT_waccs)}, Kendall's Tau: {np.mean(NoisyBT_taus)} ± {np.std(NoisyBT_taus)}")

FactorBT -- Accuracy: 0.6947212219238281 ± 0.0, Weighted Accuracy: 0.7558472752571106 ± 0.0, Kendall's Tau: 0.36763716675867 ± 0.0
